# Notebook 05 — Does Expression Predict Developmental Rewiring?

## The question

Between L1 and adult, the C. elegans connectome gains ~1,100 chemical edges (on the 185 neurons present in all stages). If gene expression encodes which neurons should become connected, the expression signatures of (pre, post) neuron pairs should distinguish **edges that get added** during development from **edges that are stable** from L1 onwards.

This is a *different* question from Notebook 03:
- 03 tested whether gene expression predicts static adult motif participation. Null (0 FDR survivors at N=84).
- 05 tests whether gene expression predicts which neuron pairs form new edges during development. **Change, not state.**

The biological hypothesis: if connectome wiring is genetically encoded, neuron pairs that form edges should have distinctive gene signatures distinguishing them from pairs that never connect, and *especially* from pairs that already connected at L1 vs those that only form adult connections.

## Preregistered design

**Edge categories** (defined on the 185 neurons present in all stages, using chemical connections):
- **stable**: edge exists in ≥3/4 L1 replicates AND in adult
- **added**: edge absent from 3+/4 L1 replicates AND present in adult
- (removed and rare — dropped for binary classification)

**Target task**: binary classification — given (pre, post) expression signature, predict stable vs added.

**Features**: concatenation of PCA-reduced expression for pre and post class (PC1–50 each → 100 features). This mirrors Notebook 04's PCA-50 reduction for p>>n safety.

**Pseudoreplication control**: collapse to unique **class-pair** level. Edges (ADAL, AIAL) and (ADAR, AIAR) both project to (ADA, AIA) at class level, and share identical features. Only one representative per class-pair is kept.

## Preregistered success criteria

1. **N class-pairs ≥ 200** after deduplication and restricting to classes in CeNGEN.
2. **Stable / added balance** not more extreme than 1:10 (for CV viability).
3. **5-fold CV accuracy ≥ majority-class baseline + 10 percentage points.** At roughly 60/40 split (the ratio after dedup at class level will settle somewhere), majority baseline is ~60%; bar is ≥ 70%.
4. **Upper 95% CI of CV mean accuracy > permutation null 95th percentile.** Permutation shuffles the stable/added label across class-pairs.
5. **AUC ≥ 0.65.** Complementary to accuracy; robust to class imbalance.

## Halting rule

If 3 or 4 fails: declare null. The conclusion is that gene expression at CeNGEN L4 larval stage does not predict which (class_pre, class_post) pairs gain connections between L1 and adult.

If 3 and 4 both pass: this is the first defensible positive finding in the entire rebuilt pipeline, and becomes the foundation of Paper 1.

In [1]:
import sys, time
from pathlib import Path
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DERIVED
from lib.reference import load_nt_reference

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

RNG = np.random.default_rng(42)

DEV = DERIVED / 'developmental'

# Load each stage
STAGES = ['L1_1','L1_2','L1_3','L1_4','L2','L3','adult']
stage_data = {}
for s in STAGES:
    d = np.load(DEV / f'connectome_{s}.npz', allow_pickle=True)
    neurons = np.array([str(n) for n in d['neurons']])
    stage_data[s] = {
        'neurons': neurons,
        'chem': d['chem_adj'],
        'n2i': {n: i for i, n in enumerate(neurons)},
    }
    print(f'{s}: {len(neurons)} neurons, {int((d["chem_adj"]>0).sum())} chem edges')

L1_1: 187 neurons, 775 chem edges
L1_2: 194 neurons, 986 chem edges
L1_3: 198 neurons, 1012 chem edges
L1_4: 204 neurons, 1136 chem edges
L2: 211 neurons, 1515 chem edges
L3: 216 neurons, 1525 chem edges
adult: 222 neurons, 2191 chem edges


## Step 1 — Define edge changes on common neurons

In [2]:
# Common-neuron subset across ALL stages
common = sorted(set(stage_data['L1_1']['neurons']).intersection(
    *[set(stage_data[s]['neurons']) for s in STAGES[1:]]
))
print(f'Common neurons (present in all 7 stages): {len(common)}')

def edge_set(stage, restrict=None):
    n = stage_data[stage]['neurons']
    chem = stage_data[stage]['chem']
    restrict_idx = set(range(len(n))) if restrict is None else {stage_data[stage]['n2i'][x] for x in restrict}
    es = set()
    for i in restrict_idx:
        for j in restrict_idx:
            if i != j and chem[i, j] > 0:
                es.add((n[i], n[j]))
    return es

L1_stages = ['L1_1','L1_2','L1_3','L1_4']
adult_edges = edge_set('adult', common)

# Count edges in each L1 replicate on common-neuron subset
edge_l1_count = {}
for s in L1_stages:
    es = edge_set(s, common)
    for e in es:
        edge_l1_count[e] = edge_l1_count.get(e, 0) + 1

# Edge categories
L1_majority = set(e for e, c in edge_l1_count.items() if c >= 3)
L1_none     = set(e for e, c in edge_l1_count.items() if c <= 1)  # absent in 3+/4 L1 replicates

stable_edges  = L1_majority & adult_edges
added_edges   = adult_edges - L1_majority   # present in adult, absent in majority of L1

# Negative class: edges that NEVER exist (absent in L1 majority AND absent in adult)
all_ordered_pairs = {(a, b) for a in common for b in common if a != b}
never_edges = all_ordered_pairs - L1_majority - adult_edges

print(f'Stable edges (L1 majority ∩ adult):       {len(stable_edges)}')
print(f'Added edges (adult only, absent in L1):    {len(added_edges)}')
print(f'Never-edges (absent in both):              {len(never_edges)}')

Common neurons (present in all 7 stages): 185
Stable edges (L1 majority ∩ adult):       653
Added edges (adult only, absent in L1):    1099
Never-edges (absent in both):              32245


## Step 2 — Map edges to CeNGEN class pairs + collapse pseudoreplication

In [3]:
# Load CeNGEN expression and neuron->class mapping
mapping_df = pd.read_csv(DERIVED / 'expression_neuron_mapping.csv')
neuron_to_class = dict(zip(mapping_df['witvliet_name'], mapping_df['cengen_class']))

expr_data = np.load(DERIVED / 'expression_tpm.npz', allow_pickle=True)
neurons_e = expr_data['neurons']
tpm = expr_data['tpm']

# Build class-level expression (one row per CeNGEN class)
class_to_expr = {}
for i, nm in enumerate(neurons_e):
    cls = neuron_to_class.get(str(nm))
    if isinstance(cls, str) and cls not in class_to_expr and not np.all(np.isnan(tpm[i])):
        class_to_expr[cls] = tpm[i]

print(f'CeNGEN classes with expression: {len(class_to_expr)}')

def edge_to_class_pair(e):
    pre, post = e
    cp = neuron_to_class.get(pre)
    cq = neuron_to_class.get(post)
    if isinstance(cp, str) and isinstance(cq, str) and cp in class_to_expr and cq in class_to_expr:
        return (cp, cq)
    return None

def collapse(edges):
    pairs = set()
    for e in edges:
        cp = edge_to_class_pair(e)
        if cp is not None:
            pairs.add(cp)
    return pairs

stable_pairs = collapse(stable_edges)
added_pairs  = collapse(added_edges)
never_pairs  = collapse(never_edges)

# Remove any overlap between stable and added (can happen because 4 neurons may collapse:
# one L/R pair is stable, the other L/R pair of the same class pair is added)
overlap_sa = stable_pairs & added_pairs
print(f'Stable class-pairs:      {len(stable_pairs)}')
print(f'Added class-pairs:       {len(added_pairs)}')
print(f'Overlap stable∩added:    {len(overlap_sa)} (class-pairs with mixed behavior — drop)')

stable_pairs -= overlap_sa
added_pairs -= overlap_sa
never_pairs -= (stable_pairs | added_pairs)  # clean never-set

print(f'\nAfter dedup:')
print(f'  stable class-pairs: {len(stable_pairs)}')
print(f'  added class-pairs:  {len(added_pairs)}')
print(f'  never class-pairs:  {len(never_pairs)}')

CeNGEN classes with expression: 84
Stable class-pairs:      238
Added class-pairs:       624
Overlap stable∩added:    97 (class-pairs with mixed behavior — drop)

After dedup:
  stable class-pairs: 141
  added class-pairs:  527
  never class-pairs:  4471


## Step 3 — Build features: concatenated PC-50 expression per (pre, post) class pair

In [4]:
# PCA on class-level expression (same reduction as Nb 04)
classes_list = sorted(class_to_expr.keys())
X_classes = np.stack([np.log1p(class_to_expr[c]) for c in classes_list])
print(f'class expression matrix: {X_classes.shape}')

N_PCA = 50
pca_model = PCA(n_components=N_PCA, random_state=42).fit(X_classes)
X_pca = pca_model.transform(X_classes)
class_to_pc = {c: X_pca[i] for i, c in enumerate(classes_list)}
print(f'PCA: {X_classes.shape} -> {X_pca.shape}  (var explained {pca_model.explained_variance_ratio_.sum():.2%})')

def pair_features(pair):
    cp, cq = pair
    return np.concatenate([class_to_pc[cp], class_to_pc[cq]])

# Build binary (stable vs added) dataset
X_stable = np.stack([pair_features(p) for p in stable_pairs])
X_added  = np.stack([pair_features(p) for p in added_pairs])

X_all = np.concatenate([X_stable, X_added])
y_all = np.concatenate([np.zeros(len(X_stable)), np.ones(len(X_added))])  # 0=stable, 1=added
pair_names = list(stable_pairs) + list(added_pairs)

print(f'\nDataset: X={X_all.shape}, y={y_all.shape}')
print(f'  stable: {(y_all==0).sum()}, added: {(y_all==1).sum()}')
print(f'  majority baseline: {max((y_all==0).mean(), (y_all==1).mean()):.3f}')

class expression matrix: (84, 13669)


PCA: (84, 13669) -> (84, 50)  (var explained 80.37%)

Dataset: X=(668, 100), y=(668,)
  stable: 141, added: 527
  majority baseline: 0.789


## Step 4 — Train classifier with 5-fold CV + permutation null

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def cv_metrics(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs, aucs = [], []
    for tr, te in skf.split(X, y):
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1))
        model.fit(X[tr], y[tr])
        accs.append(model.score(X[te], y[te]))
        aucs.append(roc_auc_score(y[te], model.predict_proba(X[te])[:, 1]))
    return np.array(accs), np.array(aucs)

accs, aucs = cv_metrics(X_all, y_all)
print(f'CV accuracy (5 folds): {[f"{a:.3f}" for a in accs]}')
print(f'  mean: {accs.mean():.3f}  std: {accs.std():.3f}')
print(f'CV AUC      (5 folds): {[f"{a:.3f}" for a in aucs]}')
print(f'  mean: {aucs.mean():.3f}  std: {aucs.std():.3f}')

# Permutation null
N_PERM = 100
t0 = time.time()
null_accs, null_aucs = [], []
for i in range(N_PERM):
    y_perm = RNG.permutation(y_all)
    a, u = cv_metrics(X_all, y_perm, random_state=42+i)
    null_accs.append(a.mean()); null_aucs.append(u.mean())
    if (i+1) % 20 == 0:
        print(f'  perm {i+1}/{N_PERM} done ({time.time()-t0:.1f}s)')
null_accs = np.array(null_accs); null_aucs = np.array(null_aucs)
print(f'\nPermutation null accuracy: mean={null_accs.mean():.3f}, 95pct={np.percentile(null_accs, 95):.3f}, max={null_accs.max():.3f}')
print(f'Permutation null AUC:      mean={null_aucs.mean():.3f}, 95pct={np.percentile(null_aucs, 95):.3f}, max={null_aucs.max():.3f}')

CV accuracy (5 folds): ['0.754', '0.799', '0.754', '0.789', '0.744']
  mean: 0.768  std: 0.022
CV AUC      (5 folds): ['0.628', '0.620', '0.613', '0.610', '0.608']
  mean: 0.616  std: 0.007


  perm 20/100 done (0.3s)


  perm 40/100 done (0.7s)


  perm 60/100 done (1.0s)


  perm 80/100 done (1.4s)


  perm 100/100 done (1.7s)

Permutation null accuracy: mean=0.752, 95pct=0.770, max=0.774
Permutation null AUC:      mean=0.497, 95pct=0.552, max=0.578


## Step 5 — Preregistered criteria check

In [6]:
N_pairs = len(y_all)
majority_baseline = max((y_all==0).mean(), (y_all==1).mean())
minority_ratio = min((y_all==0).sum(), (y_all==1).sum()) / max((y_all==0).sum(), (y_all==1).sum())
acc_mean, acc_std = accs.mean(), accs.std()
auc_mean, auc_std = aucs.mean(), aucs.std()
acc_sem = acc_std / np.sqrt(len(accs))
acc_upper = acc_mean + 1.96 * acc_sem

checks = [
    ('1 N_class_pairs >= 200',                 N_pairs >= 200,                            f'N={N_pairs}'),
    ('2 class balance not more extreme than 1:10', minority_ratio >= 0.1,                 f'ratio={minority_ratio:.2f}'),
    ('3 accuracy >= majority baseline + 0.10', acc_mean >= majority_baseline + 0.10,      f'acc={acc_mean:.3f}, base={majority_baseline:.3f}'),
    ('4 acc upper-CI > null 95pct',            acc_upper > np.percentile(null_accs, 95),  f'upper={acc_upper:.3f}, null95={np.percentile(null_accs, 95):.3f}'),
    ('5 AUC >= 0.65',                          auc_mean >= 0.65,                          f'AUC={auc_mean:.3f}'),
]

print('PREREGISTERED CRITERIA')
print('=' * 70)
all_pass = True
for label, passed, detail in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {label:48s}  {detail}')
    if not passed: all_pass = False
print('=' * 70)
print(f'ALL CRITERIA PASS: {all_pass}')

if all_pass:
    verdict = 'POSITIVE — expression predicts developmental rewiring above chance'
elif acc_mean > majority_baseline and auc_mean > 0.55:
    verdict = 'WEAK POSITIVE — signal above chance but below preregistered bar'
else:
    verdict = 'NULL — expression does not predict which class-pairs form adult-specific edges'
print(f'\nVERDICT: {verdict}')

PREREGISTERED CRITERIA
  [PASS] 1 N_class_pairs >= 200                            N=668
  [PASS] 2 class balance not more extreme than 1:10        ratio=0.27
  [FAIL] 3 accuracy >= majority baseline + 0.10            acc=0.768, base=0.789
  [PASS] 4 acc upper-CI > null 95pct                       upper=0.787, null95=0.770
  [FAIL] 5 AUC >= 0.65                                     AUC=0.616
ALL CRITERIA PASS: False

VERDICT: NULL — expression does not predict which class-pairs form adult-specific edges


## Step 6 — Save artifacts + final summary

In [7]:
# Save per-pair labels (for inspection / follow-up)
label_df = pd.DataFrame({
    'pre_class': [p[0] for p in pair_names],
    'post_class': [p[1] for p in pair_names],
    'label': ['stable' if l == 0 else 'added' for l in y_all],
})
label_df.to_csv(DERIVED / 'nb05_pair_labels.csv', index=False)

# Final summary
summary = pd.DataFrame([{
    'N_class_pairs': N_pairs,
    'n_stable': int((y_all == 0).sum()),
    'n_added': int((y_all == 1).sum()),
    'majority_baseline': majority_baseline,
    'cv_accuracy_mean': acc_mean, 'cv_accuracy_std': acc_std,
    'cv_accuracy_upper_ci': acc_upper,
    'cv_auc_mean': auc_mean, 'cv_auc_std': auc_std,
    'null_acc_95pct': float(np.percentile(null_accs, 95)),
    'null_auc_95pct': float(np.percentile(null_aucs, 95)),
    'verdict': verdict,
}])
summary.to_csv(DERIVED / 'nb05_final_summary.csv', index=False)
print(summary.T.to_string())
print(f'\nArtifacts saved to {DERIVED}')

                                                                                                   0
N_class_pairs                                                                                    668
n_stable                                                                                         141
n_added                                                                                          527
majority_baseline                                                                           0.788922
cv_accuracy_mean                                                                            0.767961
cv_accuracy_std                                                                             0.021715
cv_accuracy_upper_ci                                                                        0.786995
cv_auc_mean                                                                                 0.615684
cv_auc_std                                                                                 